# Reverse engineering LULC elements for pre-mapathon modular reference data

In [27]:
import ee

ee.Authenticate() 
ee.Initialize()

## Set seed for constant randomization

In [28]:
import random
import numpy as np

RANDOM_SEED      = 999

if RANDOM_SEED is not None:
    random.seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)

# Upload class-labelled points

In [29]:
# Upload class-labelled reference data
import geopandas as gpd

INPUT_SHAPEFILE  = "../data/modular_mapping_approach/kalbar_test/kalbar_lulc_points_new.shp"

gdf = gpd.read_file(INPUT_SHAPEFILE)

gdf.head()

,CID,ID,LULC_24,geometry
0,23,1,Hutan lahan kering primer,POINT (113.37012 0.90349)
1,23,1,Hutan lahan kering primer,POINT (112.33964 1.38568)
2,23,1,Hutan lahan kering primer,POINT (113.1101 -0.41039)
3,23,1,Hutan lahan kering primer,POINT (113.55125 1.15382)
4,23,1,Hutan lahan kering primer,POINT (113.30186 1.32283)


# Upload CSV of the randomization ruleset per class

In [30]:
import pandas as pd

RULES_CSV = "../data/modular_mapping_approach/kalbar_test/attribute_random_rules_kalbar.csv"
CLASS_FIELD = "LULC_24"          # field in the shapefile holding the class name

rules_df = pd.read_csv(RULES_CSV, dtype=str, keep_default_na=False)
rules_df = rules_df.set_index(CLASS_FIELD)

# Build {class_name: {column_name: rule_string}} from the CSV, dropping empty cells
ATTRIBUTE_RULES = {
    class_name: {col: val for col, val in row.items() if str(val).strip() != ""}
    for class_name, row in rules_df.to_dict(orient="index").items()
}

# Sanity check: list all attribute columns that will be created
ATTRIBUTE_COLUMNS = sorted({col for rules in ATTRIBUTE_RULES.values() for col in rules})
print(f"{len(ATTRIBUTE_RULES)} classes loaded from '{RULES_CSV}', "
      f"{len(ATTRIBUTE_COLUMNS)} attribute columns will be added:")
print(ATTRIBUTE_COLUMNS)

19 classes loaded from '../data/modular_mapping_approach/kalbar_test/attribute_random_rules_kalbar.csv', 15 attribute columns will be added:
['agricultural_activity', 'bareSoil_cover', 'builtup_cover', 'herb_cover', 'herb_horizontal_spreading', 'palm_cover', 'palm_horizontal_spreading', 'shrub_cover', 'shrub_horizontal_spreading', 'species_name', 'timber_extraction', 'tree_cover', 'tree_horizontal_spreading', 'waterbody_cover', 'woody_leaf_phenology']


# Check shapefile and csv class fields

## Sampling helper functions

In [31]:
import re
import random

_RANGE_RE = re.compile(r"^\s*(-?\d+(?:\.\d+)?)\s*-\s*(-?\d+(?:\.\d+)?)\s*$")


def sample_rule(rule):
    """Draw one value from a single rule string.

    - "low-high"           -> random float uniformly in [low, high], rounded to 1 decimal
    - "optionA/optionB/..." -> random.choice of the options
    - "fixedValue"          -> returned as-is
    """
    rule = str(rule).strip()

    m = _RANGE_RE.match(rule)
    if m:
        low, high = float(m.group(1)), float(m.group(2))
        if low > high:
            low, high = high, low
        return round(random.uniform(low, high), 1)

    if "/" in rule:
        options = [o.strip() for o in rule.split("/") if o.strip()]
        return random.choice(options)

    return rule


def sample_class_attributes(class_name, rules_table=ATTRIBUTE_RULES):
    """Return a dict of {column: sampled_value} for a given class_name.

    Unrecognized classes return all-NaN/None values (and print a one-time warning
    via the caller) so the row is preserved but flagged for manual review.
    """
    if class_name not in rules_table:
        return {col: None for col in ATTRIBUTE_COLUMNS}
    return {col: sample_rule(rule) for col, rule in rules_table[class_name].items()}


In [32]:
assert CLASS_FIELD in gdf.columns, (
    f"'{CLASS_FIELD}' not found in shapefile columns: {list(gdf.columns)}. "
    "Update CLASS_FIELD in the CONFIG cell to match your field name."
)

unmatched = sorted(set(gdf[CLASS_FIELD].unique()) - set(ATTRIBUTE_RULES.keys()))
if unmatched:
    print("WARNING: the following class_name values in the shapefile have no matching "
          "rule and will get empty attribute columns:")
    for c in unmatched:
        print(f"  - {c!r}")
else:
    print("All class_name values in the shapefile have matching rules.")


All class_name values in the shapefile have matching rules.


# Generate random values to the columns per points

In [33]:
sampled_rows = gdf[CLASS_FIELD].apply(sample_class_attributes)
attr_df = pd.DataFrame(list(sampled_rows), index=gdf.index)

# Merge sampled attributes into the original GeoDataFrame (geometry stays untouched)
gdf_out = gdf.join(attr_df)
gdf_out.head()

,CID,ID,LULC_24,geometry,waterbody_cover,bareSoil_cover,builtup_cover,herb_horizontal_spreading,herb_cover,palm_horizontal_spreading,palm_cover,shrub_horizontal_spreading,shrub_cover,species_name,timber_extraction,tree_cover,tree_horizontal_spreading,woody_leaf_phenology,agricultural_activity
0,23,1,Hutan lahan kering primer,POINT (113.37012 0.90349),3.9,0,0,unevenlySpread,0.8,unevenlySpread,8.7,unevenlySpread,5.7,others,No,94.9,unevenlySpread,evergreen,No
1,23,1,Hutan lahan kering primer,POINT (112.33964 1.38568),4.4,0,0,unevenlySpread,6.5,unevenlySpread,6.4,unevenlySpread,6.4,others,No,98.2,unevenlySpread,evergreen,No
2,23,1,Hutan lahan kering primer,POINT (113.1101 -0.41039),0.7,0,0,unevenlySpread,9.7,unevenlySpread,0.5,unevenlySpread,7.8,others,No,94.0,unevenlySpread,evergreen,No
3,23,1,Hutan lahan kering primer,POINT (113.55125 1.15382),4.2,0,0,unevenlySpread,4.0,unevenlySpread,2.7,unevenlySpread,9.6,others,No,96.4,unevenlySpread,evergreen,No
4,23,1,Hutan lahan kering primer,POINT (113.30186 1.32283),0.9,0,0,unevenlySpread,6.8,unevenlySpread,1.6,unevenlySpread,8.9,others,No,95.5,unevenlySpread,deciduous,No


# Save the output to local

In [34]:
from pathlib import Path

OUTPUT_SHAPEFILE = "../data/temp/kalbar_test_revengineer.shp"

Path(OUTPUT_SHAPEFILE).parent.mkdir(parents=True, exist_ok=True)
gdf_out.to_file(OUTPUT_SHAPEFILE)  # Note: .shp truncates field names to 10 chars (see Notes below)
print(f"Saved {len(gdf_out)} points with {len(ATTRIBUTE_COLUMNS)} new attribute columns to:")
print(OUTPUT_SHAPEFILE)

Saved 801 points with 15 new attribute columns to:
../data/temp/kalbar_test_revengineer.shp


C:\Users\widijanto\AppData\Local\Temp\ipykernel_45816\327497730.py:6: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf_out.to_file(OUTPUT_SHAPEFILE)  # Note: .shp truncates field names to 10 chars (see Notes below)
c:\Users\widijanto\AppData\Local\miniconda3\envs\luma-ge\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'waterbody_cover' to 'waterbody_'
  ogr_write(
c:\Users\widijanto\AppData\Local\miniconda3\envs\luma-ge\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'bareSoil_cover' to 'bareSoil_c'
  ogr_write(
c:\Users\widijanto\AppData\Local\miniconda3\envs\luma-ge\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'builtup_cover' to 'builtup_co'
  ogr_write(
c:\Users\widijanto\AppData\Local\miniconda3\envs\luma-ge\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'herb_horizont